In [1]:
import pandas as pd
import os
from pathlib import Path
import shutil
import re
import unicodedata

Crear directorio `csv_cleaned`

In [12]:
root = Path.cwd().parent

In [13]:
Path(root / "csv_cleaned").mkdir(exist_ok=True)

Cargar archivos csv

In [ ]:
for folder in Path(root / "csv_enunciados").iterdir():
    if not folder.is_dir :
        asignatura = folder.name.split(" - ")[0].replace(" ", "_").lower()
        for file in folder.iterdir():
            tema = file.stem
            shutil.copy(file, Path(root / "csv_cleaned") / f"{asignatura}_{tema}.csv")

In [4]:
def create_id(row):
    id_string = f"{row["asignatura"]}_{row["año"]}_{row["convocatoria"]}"
    if not pd.isna(row["ejercicio"]):
        id_string += f"_{row["ejercicio"]}"
    if not pd.isna(row["opcion"]):
        id_string += f"_{row["opcion"]}"
    return id_string

In [5]:
HEADER = re.compile(r"""
    \b(?:F[IÍ]SICA|QU[IÍ]MICA|MATEM[AÁ]TICAS)   # asignatura (con o sin tilde)
    [A-ZÁÉÍÓÚÑ\s.]*?                            # resto del nombre: II, APLICADAS A LAS CC.SS. II...
    \.\s*(?:19|20)\d{2}\.                       # año
    (?:\s*\(?[A-ZÁÉÍÓÚÑ0-9]+\b\)?\.?)*          # convocatoria, ejercicio, opción...
""", re.VERBOSE)

def remove_exercise_info(texto):
    texto = HEADER.sub(" ", texto)
    return re.sub(r"\s+", " ", texto).strip()

In [6]:
def insert_new_line(statement):
    new_st = statement
    for point in ["a", "b", "c", "d"]:
        new_st = new_st.replace(f" {point})", f"\n{point})")
    return new_st

In [14]:
def clean_csv(file):
    df = pd.read_csv(file, encoding='utf-8')
    df_sin_metadatos = df[df.aviso == "sin metadatos"]
    df = df[df.aviso.isna()]

    # extract subject and topic
    asignatura = file.stem.split("_")[0]
    tema = "_".join(file.stem.split("_")[1:])
    df["asignatura"] = asignatura
    df["tema"] = tema

    # year to integer
    df.año = df.año.astype(int)

    # create id
    df["id"] = df.apply(lambda row: create_id(row), axis = 1)
    
    # rearrange columns
    df = df[["id"] + list(df.columns[:-2])]

    # remove year line from statement
    df["enunciado"] = df["enunciado"].apply(remove_exercise_info)

    # remove gravity from physics statements
    if asignatura == "física":
        df["enunciado"] = df.enunciado.str.replace("g = 9,8 m s-2 ", "")
        
    # insert new line in statement points
    df["enunciado"] = df.enunciado.apply(insert_new_line)

    # normalize text
    df.enunciado = df.enunciado.apply(lambda texto : unicodedata.normalize("NFKC", texto))

    # trim multiple spaces
    df.enunciado = df.enunciado.apply(lambda texto : re.sub(r"[ \t]+", " ", texto).strip())

    # export clean csv
    output_dir = Path(root / "csv_cleaned")
    output_dir.mkdir(exist_ok=True)
    sin_metadatos_dir = output_dir / "sin_metadatos"
    sin_metadatos_dir.mkdir(exist_ok=True)
    df.to_csv(output_dir / file.name, encoding="utf-8", index = False)
    df_sin_metadatos.to_csv(sin_metadatos_dir / f"{file.name}_sin_metadatos.csv", encoding="utf-8", index = False)

In [15]:
errors = []
for file in Path(root / "csv_enunciados").iterdir():
    if file.is_file() and file.suffix == ".csv":
        try:
            clean_csv(file)
        except Exception as e:
            message = f"Error in file {file.name}: {e}"
            errors.append(message)
            print(message)

with open("errores_limpieza.txt", "w") as f:
    for error in errors:
        f.write(f"{error}\n")

In [25]:
df = pd.DataFrame()
for file in Path(root / "csv_cleaned/sin_metadatos").iterdir():
    if file.is_file() and file.suffix == ".csv":
        df_temp = pd.read_csv(file, encoding='utf-8')
        df = pd.concat([df, df_temp], ignore_index=True)

df.to_csv(Path(root / "csv_cleaned/sin_metadatos/sin_metadatos_todos.csv"), encoding="utf-8", index = False)

In [26]:
df_no_metadatos = pd.read_csv(Path(root / "csv_cleaned/sin_metadatos/sin_metadatos_todos.csv"), encoding='utf-8')

In [28]:
df_no_metadatos[["imagen"]]

,imagen
0,enunciados_PNG/2003 - Termoquímica_png/2003_te...
1,enunciados_PNG/2006 - Termoquímica_png/2006_te...
2,enunciados_PNG/2007 - Termoquímica_png/2007_te...
3,enunciados_PNG/2008 - Termoquímica_png/2008_te...
4,enunciados_PNG/2009 - Termoquímica_png/2009_te...
...,...
875,enunciados_PNG/2024 - Distribución normal y bi...
876,enunciados_PNG/2024 - Distribución normal y bi...
877,enunciados_PNG/2024 - Distribución normal y bi...
878,enunciados_PNG/2024 - Distribución normal y bi...


In [30]:
list(df_no_metadatos.imagen)

['enunciados_PNG/2003 - Termoquímica_png/2003_termoquimica_p08.png',
 'enunciados_PNG/2006 - Termoquímica_png/2006_termoquimica_p08.png',
 'enunciados_PNG/2007 - Termoquímica_png/2007_termoquimica_p08.png',
 'enunciados_PNG/2008 - Termoquímica_png/2008_termoquimica_p08.png',
 'enunciados_PNG/2009 - Termoquímica_png/2009_termoquimica_p08.png',
 'enunciados_PNG/2013 - Termoquímica_png/2013_termoquimica_p07.png',
 'enunciados_PNG/2014 - Termoquímica_png/2014_termoquimica_p08.png',
 'enunciados_PNG/2016 - Termoquímica_png/2016_termoquimica_p07.png',
 'enunciados_PNG/2001 - Equilibrio Químico_png/2001_equilibrio_quimico_p10_r2.png',
 'enunciados_PNG/2002 - Equilibrio Químico_png/2002_equilibrio_quimico_p08_r2.png',
 'enunciados_PNG/2002 - Equilibrio Químico_png/2002_equilibrio_quimico_p11.png',
 'enunciados_PNG/2004 - Equilibrio Químico_png/2004_equilibrio_quimico_p09.png',
 'enunciados_PNG/2004 - Equilibrio Químico_png/2004_equilibrio_quimico_p10.png',
 'enunciados_PNG/2008 - Equilibrio Qu